# Introduction

Several tests have been performed to assess
- the overall performance of the scheduler 
- which scheduling policies yield better carbon savings and under which conditions
- impact of configuration and external factors on the carbon saving


## Policies

The tests consider several scheduling policies (strategies) for assigning requests to slots:
- `baseline`: assigns requests as they come with the most accurate flavour, resembling the default behavior.
- `carbonshift`: runs DP solver to find optimal assignments, for both slots and flavours
- `hazard`: probability-driven selection of both slot and flavours. The name comes from the gambling "hazard" game, where two dices are rolled, in this case one for slot and one for flavour. Carbon intensity forecast acts as the probability distribution for slot choice, while each flavour has its own probability, so that on average the threshold is respected and the selection is balanced.
- `swarm/MAB`: uses multi-armed bandit to select slots, balancing exploration and exploitation. Cheapest possible slot is always preferred, respecting the error threshold.
- `swarm/ACO`: uses ant colony optimization for slot selection, each ant representing a set of assignments, with pheromone deposits on each slot. Cheapest possible slot is always preferred, respecting the error threshold.
- Extra policies, added for comparison purposes:
  - `offline/MAB`: uses MAB with full knowledge of the scenario, no online scheduling is performed.
  - `offline/ACO`: uses ACO with full knowledge, no online scheduling, allowing an ant to explore all assignments from start to finish.


## Information on the tests

Tests are conducted over three main **scenarios**:
  - `scen_1500_92`: scenario with (on average) 1500 requests per slot (randomness seed `92`) 
  - `scen_1800_230`: scenario with (on average) 1800 requests per slot (randomness seed `230`)
  - `scen_2000_120`: scenario with (on average) 2000 requests per slot (randomness seed `120`) 

A scenario a .json file comprising the requests to be scheduled (each having id, arrival time and deadline) and the carbon intensity forecast values for each slot.
Each scenario has 72 time slots.
Essentialy it contains all input data for a test run.\\
The distribution of requests over slots follows the input pattern specified by the test. That is, each scenario may have different load waves, depending on the pattern chosen for a given test run.



The maximum **deadline** slack for requests is set to 14 slots, matching the future window for possible assignments, which is also 14 slots.\
Time slots are meant to be considered **hourly**, so that each slot represents one hour.\
Given that the carbon intensity pattern presents peaks and valleys every 12 hours following the light and dark cycles of the day, with 14 hourly slots, the scheduling of requests can take advantage of the whole variation of the pattern, always seeing both a valley and a peak (not two valleys nor two peaks).\
Using 30min slots (as in previous Carbonshift literature) would double the number of slots in the same 14-hour window to still be able to exploit CI variations, making the tests even more computationally intensive. For testing purposes, we can stick to hourly slots.

The moving **window** is `[-12,+14]` plus another 12 slots in the past with decaying weights applied (from 12/13 to 1/13).
The average error must be kept below the threshold (`4.0` unless otherwise specified), both globally and within each moving window.

Three execution **flavours** are considered:
| flavour | error | exec time |
|---------|-------|-----------|
| Accurate | 0.0 | 60s |
| Balanced | 2.5 | 30s |
| Fast  | 5.0 | 10s |

Beam **pruning** with 3500 retained states is applied where `batch_size >= 8`.

**Cap levels** define the multiplying factor applied to CI when exceeding an amount of requests assigned to a time slot. That is, requests assigned beyond the cap level will have their carbon intensity cost multiplied by the corresponding factor.\
*All policies* take the defined cap levels into account when scheduling requests, except for `baseline`.\
Cap levels are defined in the `capacity_tiers` section of each scenario, and are varied only for specific test runs aimed at assessing their impact on scheduling.


#### Time slots
Each scenario has 72 time slots. That is, there are requests arriving in each of the 72 slots.\
However, the scheduler may place requests beyond the 72th slot, so the 72 slots are not a hard cap for assigning requests.\
Requests placed beyond the 72th slot are allowed, but not counted in the metrics. Slots beyond are inherently less loaded, and would not represent typical conditions. On the other hand, ignoring the slots leads to trimming a variable amount of requests for each run, which however, is more or less always the same.

Using the 72 slots as a hard cap led to having the last few slots to be overloaded, crossing the threshold and hence yielding higher cost than expected.


### Metrics

The primary metrics used to evaluate the performance of the scheduling policies are:
- `carb_saving`: measures the reduction in carbon emissions achieved by the scheduling policy.
- `solver_ms`: the average time taken by the solver to compute the schedule, in milliseconds.
- `elapsed_s`: the total elapsed time for the test run, in seconds, accounting also for any overhead outside the solver computation.

There are some execution parameters which have been varied in the experiments, in order to study their impact on the performance metrics. These include:
- `paral`: the number of parallel solver threads used by the scheduler.
- `batch`: the number of requests processed in each batch by the scheduler. 
  - Tests have different configurations, this is in `[1,4,6,8,22]` for comprehensive tests with the Carbonshift DP solver. In case of online policies, it goes up to `[1,8,22,48,96]`. Smaller/specific tests may use fewer batch sizes.
- `threshold`: error threshold applied both globally and within each moving window.
- `cap_levels`: the defined cap levels and their corresponding multiplying factors. In general 2 cap levels are defined: one for moderate exceedance and one triggering strong rebound effect.
```json
      "capacity_tiers": [
        {
          "max_requests": 2000,
          "multiplier": 1.0
        },
        {
          "max_requests": 3000,
          "multiplier": 1.5
        },
        {
          "max_requests": null,
          "multiplier": 5.0
        }
      ]
```
this reads as: "up to 2000 requests, the multiplier is 1.0; up to 3000 requests, the multiplier is 1.5; beyond 3000 requests, the multiplier is 5.0."



## Table of tests


- PO : Phase opposition
- IP : In phase
- PS : Phase shift
- FLAT : Flat arrivals


| Done | index | Pattern | Title | Description | Features | Expected outcome | Outcome |
|---|---|---|---|---|---|---|---|
| [X] | 00 | IP   | Tuning DP | Tune `batch_size` | DP only | Best N=4/6 | Best N=1 due to no fallback occurring |
| [X] | 01 | IP   | Tuning DP | Tune `batch_size`, parallelism | DP only | Parallelism=??? yields top results | ... |
| [X] | 02 | IP   | Tuning DP | Tune `batch_size` with no parallelism| DP only | ... | ... |
| [X] | 03 | IP   | Tuning DP | `batch_size=1` & vary parallelism| DP only | parallelism slows down | ... |
| [X] | 10 | IP   | Tuning PerRequest & Swarm | Tune batch sizes for ACO/MAB/Hazard/singleton_best | Swarm only | ... | ... |
| [X] | 11 | IP   | Tuning ACO | Tune n_ants & n_iter for ACO | ACO only | ... | ... |
| [X] | 12 | IP   | Tuning ACO | Tune n_ants & n_iter for ACO | ACO only | ... | ... |
| [X] | 13 | IP   | Tuning ACO | Tune n_ants & n_iter for ACO | ACO only | ... | ... |
| [X] | 14 | IP   | Paral on PerRequest & Swarm| Check how parallelism impacts on best tuned | PerRequest & Swarm | ... | ... |
| [X] | 20 | IP   | CAP1 | Set cap tiers to 1x & 2x     | DP best / accurate only | ... | ... |
| [X] | 21 | IP   | CAP2 | Set cap tiers to 1.5x & 2x   | DP best / accurate only | ... | ... |
| [X] | 22 | IP   | CAP3 | Set cap tiers to 1x & 1.5x  | DP best / accurate only | ... | ... |
| [X] | 23 | IP   | CAP4 | Set cap tiers to 1x & 1.75x  | DP best / accurate only | ... | ... |
| [ ] | 30 | IP   | Full compare | Compare all strategies | Best tuned settings for each strategy | DP should win | ... |
| [ ] | 31 | PO   | Full compare | Compare all strategies | Best tuned settings for each strategy | DP should win | ... |
| [ ] | 32 | PS   | Full compare | Compare all strategies | Best tuned settings for each strategy | DP should win | ... |
| [ ] | 33 | FLAT | Full compare | Compare all strategies | Best tuned settings for each strategy | DP should win | ... |
| [ ] | 40 | PO   | Planet v Profit | Compare all strategies on varying error thresholds | Best tuned settings for each strategy | Threshold is key in saving | ... |
| [ ] | 50 | PO   | PhaseOpposition | Set cap tiers to best factors | DP best / accurate only | ... | ... |
| [ ] | 51 | IP   | InPhase | Set cap tiers to best factors | DP best / accurate only | ... | ... |
| [ ] | 52 | IP   | PhaseShift | Set cap tiers to best factors | DP best / accurate only | ... | ... |


# Imports

In [57]:
from pathlib import Path
import json
import pandas as pd
import re
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

plt.rcParams['figure.dpi'] = 85 
REPO_ROOT = Path().resolve()
RESULTS_DIR = REPO_ROOT / "results"

comparison_metric_sources = {
    "elapsed_s": "elapsed_s",
    "carb_saving": "carb_saving",
    "error": "error",
    "solver_ms": "solver_ms",
    "carb_cost": "carb_cost",
}


def compare_metrics_wide(
    data,
    *,
    row_keys,
    comparison_column,
    comparison_values,
    metric_sources,
    filters=None,
    labels=None,
    key_renames=None,
):
    """Pivot selected metrics into ordered columns for each comparison value."""
    filters = filters or {}
    required_columns = [
        *row_keys,
        comparison_column,
        *metric_sources.values(),
        *filters,
    ]
    missing_columns = [column for column in required_columns if column not in data.columns]
    if missing_columns:
        raise ValueError(f"Comparison data is missing columns: {missing_columns}")

    selected = data
    for column, value in filters.items():
        if isinstance(value, (list, tuple, set, frozenset)):
            selected = selected.loc[selected[column].isin(value)]
        else:
            selected = selected.loc[selected[column].eq(value)]
    selected = selected.loc[selected[comparison_column].isin(comparison_values)].copy()
    if selected.empty:
        raise ValueError("No rows found for the requested comparison values and filters")

    duplicate_columns = [*row_keys, comparison_column]
    duplicate_rows = selected.duplicated(duplicate_columns, keep=False)
    if duplicate_rows.any():
        details = selected.loc[duplicate_rows, duplicate_columns].drop_duplicates()
        raise ValueError(
            "Expected one row per comparison key; duplicates found:\n"
            + details.to_string(index=False)
        )

    key_renames = key_renames or {}
    metric_renames = {
        source: metric for metric, source in metric_sources.items()
    }
    selected = selected.rename(columns={**key_renames, **metric_renames})
    output_keys = [key_renames.get(column, column) for column in row_keys]
    metrics = list(metric_sources)
    wide = selected.pivot(
        index=output_keys,
        columns=comparison_column,
        values=metrics,
    ).reindex(
        columns=pd.MultiIndex.from_product([metrics, comparison_values])
    )

    labels = labels or {}
    wide.columns = [
        f"{metric}_{labels.get(value, value)}"
        for metric, value in wide.columns
    ]
    result = wide.reset_index().sort_values(output_keys).reset_index(drop=True)
    comparison_columns = [
        f"{metric}_{labels.get(value, value)}"
        for metric in metrics
        for value in comparison_values
    ]
    return result.loc[:, [*output_keys, *comparison_columns]]

In [58]:
# # Makes table interactive in html
# from itables import init_notebook_mode
# init_notebook_mode(all_interactive=True)


# # Fix table alignment in HTML exported notebook
# from IPython.display import display, HTML

# display(HTML("""
# <style>
#     /* Individua i contenitori delle celle Markdown in VSCode/Jupyter */
#     .rendered_html, 
#     .jp-RenderedHTMLCommon, 
#     div[data-mime-type="text-markdown"] {
#         display: flex !important;
#         flex-direction: column !important;
#         align-items: flex-start !important;
#     }

#     /* Ripristina i margini corretti per la tabella */
#     .rendered_html table, 
#     .jp-RenderedHTMLCommon table,
#     div[data-mime-type="text-markdown"] table {
#         margin-left: 0 !important;
#         margin-right: auto !important;
#         display: table !important;
#     }
# </style>
# """))

In [59]:
# %%capture
# disable output for this cell

# this loads and aggregates the battery test results
from save_results import aggregate_and_save
SORTING_CRITERIA = ["scenario_id", "policy", "batch", "error", "paral", "path"]
df_all = aggregate_and_save(prefix_regex=r"^T", sort_by=SORTING_CRITERIA)

Trovate 25 run directories
 - T00_sine_IP_L_dp_0_1110_1913_20
 - T01_sine_IP_L_dp_0_1114_1157_1
 - T01_sine_IP_L_dp_1_1114_1403_10
 - T01_sine_IP_L_dp_2_1114_1428_20
 - T02_sine_IP_L_dp_0_1113_0004_1
 - T03_sine_IP_L_dp_0_1115_1659_1
 - T03_sine_IP_L_dp_1_1115_1700_5
 - T03_sine_IP_L_dp_2_1115_1701_10
 - T03_sine_IP_L_dp_3_1115_1702_14
 - T03_sine_IP_L_dp_4_1115_1703_20
 - T10_sine_IP_L_online_0_1113_2330_20
 - T10_sine_IP_L_perrequest_0_1113_2330_20
 - T11_sine_IP_L_online_0_1113_1222_20
 - T12_sine_IP_L_online_0_1113_1705_20
 - T13_sine_IP_L_online_0_1113_1740_20
 - T14_sine_IP_L_online_0_1115_1746_1
 - T14_sine_IP_L_online_1_1115_1748_10
 - T14_sine_IP_L_online_2_1115_1748_20
 - T14_sine_IP_L_perrequest_0_1115_1749_1
 - T14_sine_IP_L_perrequest_1_1115_1750_10
 - T14_sine_IP_L_perrequest_2_1115_1751_20
 - T20_sine_IP_L_dp_0_1111_1620_20
 - T21_sine_IP_L_dp_0_1111_1624_20
 - T22_sine_IP_L_dp_0_1111_1622_20
 - T23_sine_IP_L_dp_0_1112_1622_20
Righe totali: 300
Colonne: ['run_id', 'scena

In [60]:
import sys

sys.path.insert(0, str(RESULTS_DIR.parent))
from nb_utils import require_output_schema

output_csv = RESULTS_DIR / "aggregated_battery_results.csv"
require_output_schema(output_csv, schema_name="carbonshift.battery-aggregate")
pd.set_option("display.max_columns", None)
df = df_all[df_all["path"].str.contains("^T")]
pd.set_option("display.max_rows", 5)
display(
    df.sort_values(
        by=SORTING_CRITERIA
    )
)



,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
0,T20,scen_1500_92_sine_IP_L,105723,95899,1,dp,24.378407,9.929,0.000000,172580.143867,0,20,0.006329,20,17.952006,228215.429815,T20_sine_IP_L_dp_0_1111_1620_20/scen_1500_92_s...
1,T21,scen_1500_92_sine_IP_L,105723,96728,1,dp,24.056880,10.000,0.000000,161051.007505,0,20,0.006371,20,17.879127,212067.936187,T21_sine_IP_L_dp_0_1111_1624_20/scen_1500_92_s...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
298,T10,scen_2000_120_sine_IP_L,146276,133817,2000,online_singleton_best,76.444157,2.350,3.951927,83809.312645,0,20,12.600183,3,1.184880,355789.906033,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_2...
299,T10,scen_2000_120_sine_IP_L,146276,132773,3000,online_singleton_best,78.081796,2.917,3.998064,77255.734088,0,20,20.824270,2,1.017971,352472.925876,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_2...


# Test 00 - Tuning DP

In [61]:
df = df_all[df_all["path"].str.contains("^T00")]
pd.set_option("display.max_rows", 15)
display(df.sort_values(by=SORTING_CRITERIA))


,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
11,T00,scen_1500_92_sine_IP_L,105723,95895,1,dp,77.157229,11.335,4.000078,52128.529802,239,20,0.015666,20,18.333197,228205.810929,T00_sine_IP_L_dp_0_1110_1913_20/scen_1500_92_s...
21,T00,scen_1500_92_sine_IP_L,105723,95920,4,dp,77.029512,47.480,4.000547,52439.343861,1346,20,32.129292,20,18.450803,228290.073447,T00_sine_IP_L_dp_0_1110_1913_20/scen_1500_92_s...
28,T00,scen_1500_92_sine_IP_L,105723,95939,6,dp,76.659059,240.304,4.000302,53303.171693,4001,20,259.821556,20,19.186612,228367.704178,T00_sine_IP_L_dp_0_1110_1913_20/scen_1500_92_s...
29,T00,scen_1500_92_sine_IP_L,105723,95961,8,dp,75.089204,566.509,3.995712,56911.279716,13297,20,821.097378,20,19.245761,228460.302948,T00_sine_IP_L_dp_0_1110_1913_20/scen_1500_92_s...
32,T00,scen_1500_92_sine_IP_L,105723,96008,22,dp,74.397038,1854.377,3.967091,58516.908337,15626,20,7112.797170,20,18.587613,228555.232695,T00_sine_IP_L_dp_0_1110_1913_20/scen_1500_92_s...
104,T00,scen_1800_230_sine_IP_L,128171,117095,1,dp,76.796413,14.206,3.997950,66564.233057,234,20,0.014515,20,18.534046,286870.439045,T00_sine_IP_L_dp_0_1110_1913_20/scen_1800_230_...
122,T00,scen_1800_230_sine_IP_L,128171,117091,4,dp,76.769153,57.457,4.003830,66639.789623,1532,20,32.617094,20,18.703884,286859.065447,T00_sine_IP_L_dp_0_1110_1913_20/scen_1800_230_...
128,T00,scen_1800_230_sine_IP_L,128171,117091,6,dp,75.946651,325.580,4.003531,69000.141175,6739,20,292.962515,20,19.332303,286862.928450,T00_sine_IP_L_dp_0_1110_1913_20/scen_1800_230_...
129,T00,scen_1800_230_sine_IP_L,128171,117093,8,dp,74.641743,704.540,3.995179,72747.769948,17338,20,848.275325,20,19.366806,286880.014579,T00_sine_IP_L_dp_0_1110_1913_20/scen_1800_230_...
132,T00,scen_1800_230_sine_IP_L,128171,117134,22,dp,74.136168,2226.664,3.973782,74230.397852,19622,20,7153.369055,20,18.842885,287004.637693,T00_sine_IP_L_dp_0_1110_1913_20/scen_1800_230_...


## Why saving does not grow along with `batch_size`

Increasing N (`batch_size`) allows to try more combinations of request placements, possibly finding more optimal schedules. However, higher N means more stale data: every worker receives an assignment snapshot of the current state; upon the commit of the solver proposed assignments, the error threshold may exceeded due to multiple sibling workers committing proposals made on outdated information. This yields following solver runs to fallback to the most accurate flavour, being incapable of finding alternative assignments which respect the error threshold, which have been already crossed.

20 workers each committing proposals for 22 requests means potentially up to 440 requests being proposed simultaneously, easily exceeding the threshold and causing fallbacks.\
Fallback is costful and completely outweighs the small potential gains from higher batch sizes.

Plus, larger N exponentially increases the combinations to consider, hence the processing time.

Furthermore, the flavours considered by the solver exhibit *forecasted* error and execution time, not real ones. In a real scenario, a request will be processed and the actual error may differ, as well as the execution time. Depending on the kind of task, they may vary significantly, easily outweighing the small benefits of larger batch sizes.
So, increasing batch size is not beneficial.


Technically, we could try to come up with different fallback strategies, but they did not seem to yield much improvement. Besides, we are talking nickels here.


## TEST 01 - Varying Batch size and parallelism

Here we vary both batch size and parallelism:
 - `batch_size = [1,2,3,4,5]`
 - `parallelism = [1,10,20]`

In [62]:
df = df_all[df_all["path"].str.contains("T01")]
df_par1 = df[df["paral"] == 1]
display(df_par1.head())

df_par10 = df[df["paral"] == 10]
display(df_par10.head())

df_par20 = df[df["paral"] == 20]
display(df_par20.head())


,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
6,T01,scen_1500_92_sine_IP_L,105723,95984,1,dp,78.172579,9.868,3.998557,52015.133343,114,1,0.012727,1,1.0,238301.782258,T01_sine_IP_L_dp_0_1114_1157_1/scen_1500_92_si...
16,T01,scen_1500_92_sine_IP_L,105723,95984,2,dp,78.617261,14.215,4.005980,50955.447819,29,1,0.170464,1,1.0,238301.782258,T01_sine_IP_L_dp_0_1114_1157_1/scen_1500_92_si...
19,T01,scen_1500_92_sine_IP_L,105723,95983,3,dp,78.707802,65.700,4.006126,50738.847712,36,1,1.723119,1,1.0,238297.835352,T01_sine_IP_L_dp_0_1114_1157_1/scen_1500_92_si...
23,T01,scen_1500_92_sine_IP_L,105723,95983,4,dp,78.663018,322.397,4.001229,50845.566452,12,1,11.998135,1,1.0,238297.835352,T01_sine_IP_L_dp_0_1114_1157_1/scen_1500_92_si...
27,T01,scen_1500_92_sine_IP_L,105723,95983,5,dp,78.869023,1742.340,4.010788,50354.661904,11,1,82.030177,1,1.0,238297.835352,T01_sine_IP_L_dp_0_1114_1157_1/scen_1500_92_si...


,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
9,T01,scen_1500_92_sine_IP_L,105723,95983,1,dp,78.092703,10.354,3.998937,52204.290952,97,10,0.013486,10,9.689440,238296.358989,T01_sine_IP_L_dp_1_1114_1403_10/scen_1500_92_s...
14,T01,scen_1500_92_sine_IP_L,105723,95984,2,dp,78.069628,7.301,3.999495,52259.869115,256,10,0.198174,10,9.639796,238299.057327,T01_sine_IP_L_dp_1_1114_1403_10/scen_1500_92_s...
18,T01,scen_1500_92_sine_IP_L,105723,95993,3,dp,78.145496,10.421,4.005839,52085.707597,332,10,2.160445,10,9.681807,238329.392908,T01_sine_IP_L_dp_1_1114_1403_10/scen_1500_92_s...
22,T01,scen_1500_92_sine_IP_L,105723,95998,4,dp,78.174026,56.478,4.000839,52022.060566,417,10,19.997617,10,9.606948,238349.319738,T01_sine_IP_L_dp_1_1114_1403_10/scen_1500_92_s...
26,T01,scen_1500_92_sine_IP_L,105723,95998,5,dp,78.463541,298.357,4.001516,51332.956927,372,10,132.362294,10,9.442437,238353.747268,T01_sine_IP_L_dp_1_1114_1403_10/scen_1500_92_s...


,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
13,T01,scen_1500_92_sine_IP_L,105723,95983,1,dp,78.055852,10.345,4.000813,52292.644839,175,20,0.013635,20,18.270554,238298.819074,T01_sine_IP_L_dp_2_1114_1428_20/scen_1500_92_s...
15,T01,scen_1500_92_sine_IP_L,105723,95983,2,dp,78.096171,7.234,4.001907,52196.184767,214,20,0.194850,20,17.307251,238297.077866,T01_sine_IP_L_dp_2_1114_1428_20/scen_1500_92_s...
17,T01,scen_1500_92_sine_IP_L,105723,96006,3,dp,78.019346,9.417,4.002302,52398.653601,929,20,2.950019,20,18.379166,238385.322516,T01_sine_IP_L_dp_2_1114_1428_20/scen_1500_92_s...
20,T01,scen_1500_92_sine_IP_L,105723,96006,4,dp,78.028825,47.927,3.999099,52372.249447,1159,20,32.481874,20,18.505508,238367.993881,T01_sine_IP_L_dp_2_1114_1428_20/scen_1500_92_s...
25,T01,scen_1500_92_sine_IP_L,105723,96017,5,dp,78.290090,271.362,3.999318,51756.020329,1141,20,229.226864,20,17.990165,238398.131487,T01_sine_IP_L_dp_2_1114_1428_20/scen_1500_92_s...


In [63]:
comparison_parallelisms = [1, 10, 20]
parallelism_comparison = compare_metrics_wide(
    df_all,
    row_keys=["scenario_id", "batch"],
    comparison_column="paral",
    comparison_values=comparison_parallelisms,
    metric_sources=comparison_metric_sources,
    filters={"run_id": "T01", "policy": "dp"},
    labels={1: "P1", 10: "P10", 20: "P20"},
    key_renames={"batch": "batch_size"},
)
display(parallelism_comparison)

,scenario_id,batch_size,elapsed_s_P1,elapsed_s_P10,elapsed_s_P20,carb_saving_P1,carb_saving_P10,carb_saving_P20,error_P1,error_P10,error_P20,solver_ms_P1,solver_ms_P10,solver_ms_P20,carb_cost_P1,carb_cost_P10,carb_cost_P20
0,scen_1500_92_sine_IP_L,1,9.868,10.354,10.345,78.172579,78.092703,78.055852,3.998557,3.998937,4.000813,0.012727,0.013486,0.013635,52015.133343,52204.290952,52292.644839
1,scen_1500_92_sine_IP_L,2,14.215,7.301,7.234,78.617261,78.069628,78.096171,4.005980,3.999495,4.001907,0.170464,0.198174,0.194850,50955.447819,52259.869115,52196.184767
2,scen_1500_92_sine_IP_L,3,65.700,10.421,9.417,78.707802,78.145496,78.019346,4.006126,4.005839,4.002302,1.723119,2.160445,2.950019,50738.847712,52085.707597,52398.653601
3,scen_1500_92_sine_IP_L,4,322.397,56.478,47.927,78.663018,78.174026,78.028825,4.001229,4.000839,3.999099,11.998135,19.997617,32.481874,50845.566452,52022.060566,52372.249447
4,scen_1500_92_sine_IP_L,5,1742.340,298.357,271.362,78.869023,78.463541,78.290090,4.010788,4.001516,3.999318,82.030177,132.362294,229.226864,50354.661904,51332.956927,51756.020329
5,scen_1800_230_sine_IP_L,1,11.571,13.670,13.767,77.410990,77.336074,77.316039,3.999194,3.999966,4.000793,0.011251,0.013879,0.014132,66347.188423,66566.255170,66626.262495
6,scen_1800_230_sine_IP_L,2,16.920,9.086,9.114,77.939487,77.386920,77.443621,4.002180,4.001147,4.002807,0.164731,0.196438,0.190661,64794.916383,66417.496581,66251.998084
7,scen_1800_230_sine_IP_L,3,71.964,12.649,11.550,78.145770,77.327649,77.300871,4.008855,4.004658,4.008842,1.555626,2.197293,2.897423,64189.031055,66591.796340,66672.635272
8,scen_1800_230_sine_IP_L,4,366.895,68.957,57.806,78.346124,77.450325,77.289149,4.025384,4.001066,4.002107,11.256265,20.284926,32.748060,63600.563317,66231.280888,66702.800257
9,scen_1800_230_sine_IP_L,5,1942.713,363.460,325.372,78.209546,77.783503,77.530818,4.009602,4.003451,4.001753,75.459816,134.660767,231.215996,64001.712824,65253.801659,65990.435993


### Parallelism effect
Parallelism leads the solver time increase, since there is more contention, but in general provides a considerable speedup on the total elapsed time (6.4x, 540% faster). Note that acquiring locks is not considered in this measure, it's just the pure solver time.\
Parallelism implies a negligible performance degration in saving (<1%).

There is not much difference between parallelism=10 and 20, perhaps due to the real physical amount of threads actually available to WSL.


Parallelism though seems uneffective for N=1. That's probably because the most work is serial anyway. 0.013ms per solve is very small, and for about 100k solves it adds up to 1.3s, out of the total 10s elapsed.
That's because the assignment committing is lock guarded, hence serial, and parallelism can't help much. The same goes for writing out results as the run progresses. In fact, parallelism and lock management end up introducing more overheap than benefit.


## T01/T03 - More testing on N=1 
T03 is another test for N=1, to ensure parallelism really does not speed up the execution and that it was not a random case what we saw before.

In [64]:
df = df_all[df_all["path"].str.contains("T03")]
display(df.sort_values(["scenario_id","paral"]))

,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
8,T03,scen_1500_92_sine_IP_L,105723,95984,1,dp,78.172579,8.976,3.998557,52015.133343,114,1,0.011279,1,1.000000,238301.782258,T03_sine_IP_L_dp_0_1115_1659_1/scen_1500_92_si...
4,T03,scen_1500_92_sine_IP_L,105723,95983,1,dp,78.015795,10.584,3.996619,52387.884439,204,5,0.014391,5,4.976283,238297.835352,T03_sine_IP_L_dp_1_1115_1700_5/scen_1500_92_si...
5,T03,scen_1500_92_sine_IP_L,105723,95985,1,dp,77.987201,10.547,3.998125,52457.652617,257,10,0.014496,10,9.768295,238305.236524,T03_sine_IP_L_dp_2_1115_1701_10/scen_1500_92_s...
12,T03,scen_1500_92_sine_IP_L,105723,95984,1,dp,78.051609,10.716,4.000797,52303.299473,235,14,0.014368,14,13.413859,238301.289618,T03_sine_IP_L_dp_3_1115_1702_14/scen_1500_92_s...
10,T03,scen_1500_92_sine_IP_L,105723,95983,1,dp,78.023074,10.519,3.999458,52370.430352,267,20,0.014169,20,18.308222,238297.342712,T03_sine_IP_L_dp_4_1115_1703_20/scen_1500_92_s...
108,T03,scen_1800_230_sine_IP_L,128171,117218,1,dp,77.410990,11.461,3.999194,66347.188423,191,1,0.011275,1,1.000000,293714.453173,T03_sine_IP_L_dp_0_1115_1659_1/scen_1800_230_s...
111,T03,scen_1800_230_sine_IP_L,128171,117218,1,dp,77.344020,13.543,4.000239,66544.169692,136,5,0.013850,5,4.980782,293715.691325,T03_sine_IP_L_dp_1_1115_1700_5/scen_1800_230_s...
109,T03,scen_1800_230_sine_IP_L,128171,117216,1,dp,77.304189,13.533,3.999561,66659.565445,212,10,0.013887,10,9.795152,293708.676815,T03_sine_IP_L_dp_2_1115_1701_10/scen_1800_230_...
112,T03,scen_1800_230_sine_IP_L,128171,117217,1,dp,77.356439,14.494,4.000700,66506.624551,148,14,0.015206,14,13.492047,293710.983389,T03_sine_IP_L_dp_3_1115_1702_14/scen_1800_230_...
105,T03,scen_1800_230_sine_IP_L,128171,117216,1,dp,77.240189,13.620,3.998388,66847.763366,255,20,0.014292,20,18.599715,293709.658587,T03_sine_IP_L_dp_4_1115_1703_20/scen_1800_230_...


## TEST 02 - Batch size with no parallelism
How much can we gain from higher batch sizes if parallelism (hence fallbacks) cannot hinder performance?
*Spoiler*: More or less none.

In [65]:
df = df_all[df_all["path"].str.contains("T02")]
display(df)

,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
7,T02,scen_1500_92_sine_IP_L,105723,95984,1,dp,78.172579,8.955,3.998557,52015.133343,114,1,0.011184,1,1.0,238301.782258,T02_sine_IP_L_dp_0_1113_0004_1/scen_1500_92_si...
24,T02,scen_1500_92_sine_IP_L,105723,95983,4,dp,78.664686,296.469,4.001229,50841.591559,12,1,11.030392,1,1.0,238297.835352,T02_sine_IP_L_dp_0_1113_0004_1/scen_1500_92_si...
30,T02,scen_1500_92_sine_IP_L,105723,95983,8,dp,78.667860,2756.135,4.000214,50834.027938,16,1,207.856832,1,1.0,238297.835352,T02_sine_IP_L_dp_0_1113_0004_1/scen_1500_92_si...
31,T02,scen_1500_92_sine_IP_L,105723,95981,16,dp,78.583897,5892.239,4.010325,51032.418300,0,1,886.823263,1,1.0,238289.941540,T02_sine_IP_L_dp_0_1113_0004_1/scen_1500_92_si...
107,T02,scen_1800_230_sine_IP_L,128171,117218,1,dp,77.410990,11.623,3.999194,66347.188423,191,1,0.011096,1,1.0,293714.453173,T02_sine_IP_L_dp_0_1113_0004_1/scen_1800_230_s...
124,T02,scen_1800_230_sine_IP_L,128171,117218,4,dp,78.345841,342.947,4.025384,63601.394562,22,1,10.519401,1,1.0,293714.453173,T02_sine_IP_L_dp_0_1113_0004_1/scen_1800_230_s...
130,T02,scen_1800_230_sine_IP_L,128171,117217,8,dp,78.032267,3289.756,3.999569,64522.116198,16,1,204.752282,1,1.0,293713.128371,T02_sine_IP_L_dp_0_1113_0004_1/scen_1800_230_s...
131,T02,scen_1800_230_sine_IP_L,128171,117215,16,dp,77.878002,7148.422,4.006185,64973.963982,16,1,888.275584,1,1.0,293707.489701,T02_sine_IP_L_dp_0_1113_0004_1/scen_1800_230_s...
210,T02,scen_2000_120_sine_IP_L,146276,132741,1,dp,78.141849,13.571,3.999669,77022.737391,48,1,0.010210,1,1.0,352375.359939,T02_sine_IP_L_dp_0_1113_0004_1/scen_2000_120_s...
223,T02,scen_2000_120_sine_IP_L,146276,132740,4,dp,78.903960,403.166,4.008795,74336.824081,32,1,10.835811,1,1.0,352373.359083,T02_sine_IP_L_dp_0_1113_0004_1/scen_2000_120_s...


Having no parallelism takes down the problem of fallbacks due to simultaneous commits. Fallbacks may occur only upon the sliding of the window, which is less frequent and is smoothed by the decayed past history.


However we can see that the benefit of N>1 with respect to N=1 is almost negligible, and surely would be even more negligible in practice, where the forecasted execution times (hence carbon cost) will be replaced by the actual ones, which (most likely) exhibit high variance.

Interestingly we can see that N=16 yields slightly lower carbon savings than N=8 or N=4. Seems counterintuitive, but actually it could be due to pruning: with higher N, the options grow exponentially, and there is a higher chance some potentially good options get pruned.

Furthermore,for N>1 there is huge overhead: elapsed time grows exponentially with N. We go from almost instantaneous solving (0.01ms) with N=1 to 880ms with N=16 for a single batch.
So, for high N, we would also have to consider the carbon cost of the overhead, which may outweigh the benefits, depending on the carbon cost of the tasks at hand.

## Conclusion

N=1 is simplest and yields the best results, and is extra fast.

# TEST 20|21|22|23 - Analyzing cap tiers

In [66]:
df = df_all[df_all["path"].str.contains("^T2")]
# Remove "_sine_IP_L" from scenario ids
df["scenario_id"] = df["scenario_id"].str.replace("_sine_IP_L", "")
pd.set_option("display.max_rows", 15)
display(df.sort_values(by=SORTING_CRITERIA))

/tmp/ipykernel_149132/1616735503.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["scenario_id"] = df["scenario_id"].str.replace("_sine_IP_L", "")


,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
0,T20,scen_1500_92,105723,95899,1,dp,24.378407,9.929,0.0,172580.143867,0,20,0.006329,20,17.952006,228215.429815,T20_sine_IP_L_dp_0_1111_1620_20/scen_1500_92_s...
1,T21,scen_1500_92,105723,96728,1,dp,24.056880,10.000,0.0,161051.007505,0,20,0.006371,20,17.879127,212067.936187,T21_sine_IP_L_dp_0_1111_1624_20/scen_1500_92_s...
2,T22,scen_1500_92,105723,96052,1,dp,34.246391,9.831,0.0,174743.037433,0,20,0.006180,20,17.711413,265754.290321,T22_sine_IP_L_dp_0_1111_1622_20/scen_1500_92_s...
3,T23,scen_1500_92,105723,95983,1,dp,27.318487,10.467,0.0,173196.326556,0,20,0.007110,20,18.083071,238294.882627,T23_sine_IP_L_dp_0_1112_1622_20/scen_1500_92_s...
100,T20,scen_1800_230,128171,117092,1,dp,23.189349,12.681,0.0,220341.539242,0,20,0.006289,20,18.206157,286863.262942,T20_sine_IP_L_dp_0_1111_1620_20/scen_1800_230_...
101,T21,scen_1800_230,128171,117740,1,dp,22.956912,12.405,0.0,203944.322775,0,20,0.005972,20,18.072744,264714.627817,T21_sine_IP_L_dp_0_1111_1624_20/scen_1800_230_...
102,T22,scen_1800_230,128171,117326,1,dp,31.947099,12.525,0.0,222062.584273,0,20,0.006046,20,17.995189,326308.769016,T22_sine_IP_L_dp_0_1111_1622_20/scen_1800_230_...
103,T23,scen_1800_230,128171,117218,1,dp,24.794831,13.527,0.0,220887.962121,0,20,0.006945,20,18.182206,293713.802724,T23_sine_IP_L_dp_0_1112_1622_20/scen_1800_230_...
200,T20,scen_2000_120,146276,132741,1,dp,23.191030,16.213,0.0,256007.878161,0,20,0.006469,20,18.339453,333304.662590,T20_sine_IP_L_dp_0_1111_1620_20/scen_2000_120_...
201,T21,scen_2000_120,146276,132846,1,dp,22.660166,15.980,0.0,237641.737330,0,20,0.006367,20,18.360228,307269.520340,T21_sine_IP_L_dp_0_1111_1624_20/scen_2000_120_...


| TEST | SCENARIO | CAP TIERS (threshold, CI multiplier) |
|---|---|---|
| 20   | scen_1500_92 | (1500, 1.5x) / (3000, 5.0x) |
| 20   | scen_1800_230 | (1800, 1.5x) / (3600, 5.0x) |
| 20   | scen_2000_120 | (2000, 1.5x) / (4000, 5.0x) |
| 21   | scen_1500_92 | (2250, 1.5x) / (3000, 5.0x) |
| 21   | scen_1800_230 | (2700, 1.5x) / (3600, 5.0x) |
| 21   | scen_2000_120 | (3000, 1.5x) / (4000, 5.0x) |
| 22   | scen_1500_92 | (1500, 1.5x) / (2250, 5.0x) |
| 22   | scen_1800_230 | (1800, 1.5x) / (2700, 5.0x) |
| 22   | scen_2000_120 | (2000, 1.5x) / (3000, 5.0x) |
| 23   | scen_1500_92 | (1500, 1.5x) / (2625, 5.0x) |
| 23   | scen_1800_230 | (1800, 1.5x) / (3150, 5.0x) |
| 23   | scen_2000_120 | (2000, 1.5x) / (3500, 5.0x) |

So essentially, cap tiers are set based on the scenario's average load. For T20 tests, the first cap matches the average load, while the second its double. For T21, the first cap is 1.5x the average and the second 2x. For T22, the first matches the average and the second 1.5x. For T23, the first matches the average and the second is 1.75x the average.
Carbon intensity multiplier are always 1.5x and 5.0x.

Highest carbon saving (~33% vs 24%) is obtained for T22.
However, this happens mainly because the baseline cost is higher for T22.
By looking at the carbon cost alone T22 is always the highest.
T23 seems to exhibit the higher saving, with an absolute carbon cost lower than T22 and comparable to T20.
T21 exhibits the lowest absolute carbon cost, given that its cap tiers are both high.

Cap tiers are not really a configurable parameter in reality, they are bound to hardware resources.

These tests allow to state that the system should be able to handle under standard conditions loads up to 1.75x the average input load, before triggering a strong rebound effect.


# TEST 10 - Tuning Online and Swarm strategies

In [67]:
df = df_all[df_all["path"].str.contains("^T1")]
# Remove "_sine_IP_L" from scenario ids
df["scenario_id"] = df["scenario_id"].str.replace("_sine_IP_L", "")
pd.set_option("display.max_rows", 5)
display(df.sort_values(by=SORTING_CRITERIA))

/tmp/ipykernel_149132/664844074.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["scenario_id"] = df["scenario_id"].str.replace("_sine_IP_L", "")


,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
33,T11,scen_1500_92,105723,101453,1,online_ant_colony,50.014356,10.789,3.855406,126767.856345,0,20,0.203392,20,18.842796,253608.530464,T11_sine_IP_L_online_0_1113_1222_20/scen_1500_...
34,T13,scen_1500_92,105723,101760,1,online_ant_colony,49.600202,10.731,3.856697,128263.518752,0,20,0.129207,20,18.554581,254492.126442,T13_sine_IP_L_online_0_1113_1740_20/scen_1500_...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
298,T10,scen_2000_120,146276,133817,2000,online_singleton_best,76.444157,2.350,3.951927,83809.312645,0,20,12.600183,3,1.184880,355789.906033,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_2...
299,T10,scen_2000_120,146276,132773,3000,online_singleton_best,78.081796,2.917,3.998064,77255.734088,0,20,20.824270,2,1.017971,352472.925876,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_2...


## Ant colony

In general ACO should perform best with higher batch_sizes, because a single "ant" corresponds to a whole solution, i.e. assignments for all the requests in the batch.
Considering a higher batch_size allows more possibilities to be explored.

However there is a limit to how much can be gained, as flushing will still occur:
- Consider that requests follow a sinusoidal pattern, with valleys below the average load and peaks up to 2x above it.\
- Consider also that the scheduler when entering a new slot, forces the flush of the requests in the queue, even if they are less than `batch_size`.\
- Thus, setting a high batch size like 1000 will imply flushing often, even if the queue is not full.\

There is also another aspect. ACO has a configurable amount of ants and iterations, which affects how many solutions it explores.\
For these experiments we stick to a maximum of `n_ants = 30` and `n_iterations = 10`, which limits the search effort to 400 solutions.\
(Note that low batch sizes impose a cap to explore fewer solutions)

Tests using `n_ants = 80` and `n_iterations = 30` bring the search effort to 2400 solutions, improving the quality but resulting in very high overhead time-wise (on average 12 seconds for a 3000 request batch)



### How many ants and how many iterations?
- T11: `n_ants = 30` and `n_iterations = 10`
- T12: `n_ants = 25` and `n_iterations = 5`
- T13: `n_ants = 10` and `n_iterations = 3`

We evaluate the performance with these configurations.


NOTE:
TODO: re-exec this. Even on a single scenario would be okay.
The following result has not been saved.
<pre style="font-size: 0.8em; line-height: 1.15;">
Wrote online-strategy output to /home/francis/unipi/carbonshift/carbonshift/tests/battery/results/T01_sine_IP_L_online_0_1112_2244_20/scen_1500_92_sine_IP_L/rust_online.
    [rust/online_ant_colony] N=1: solver_ms=0.2, carbon=125929.827, error=3.874, saving=121812.752 gCO2 (49.2%), elapsed=11.7s
    [rust/online_ant_colony] N=8: solver_ms=29.3, carbon=119021.852, error=3.800, saving=123768.986 gCO2 (51.0%), elapsed=21.7s
    [rust/online_ant_colony] N=100: solver_ms=1526.9, carbon=123350.889, error=3.774, saving=118449.686 gCO2 (49.0%), elapsed=131.4s
    [rust/online_ant_colony] N=250: solver_ms=2509.9, carbon=144624.407, error=3.775, saving=102612.044 gCO2 (41.5%), elapsed=191.8s
    [rust/online_ant_colony] N=500: solver_ms=3965.6, carbon=138590.402, error=3.924, saving=122214.981 gCO2 (46.9%), elapsed=337.5s
    [rust/online_ant_colony] N=1000: solver_ms=7042.6, carbon=125485.498, error=3.953, saving=135834.553 gCO2 (52.0%), elapsed=604.3s
    [rust/online_ant_colony] N=1500: solver_ms=8477.2, carbon=100413.235, error=3.956, saving=161089.357 gCO2 (61.6%), elapsed=716.6s
    [rust/online_ant_colony] N=2000: solver_ms=9893.8, carbon=90974.798, error=3.926, saving=163097.141 gCO2 (64.2%), elapsed=816.5s
    [rust/online_ant_colony] N=3000: solver_ms=12335.8, carbon=101097.899, error=3.920, saving=151528.065 gCO2 (60.0%), elapsed=900.1s
  Scenario scen_1500_92_sine_IP_L completed in 3759.0s
</pre>

In [68]:
# T11: n_ants = 30 and n_iterations = 10
# T12: n_ants = 25 and n_iterations = 5
# T13: n_ants = 10 and n_iterations = 3

aco_comparison = compare_metrics_wide(
    df_all,
    row_keys=["scenario_id", "batch"],
    comparison_column="run_id",
    comparison_values=["T11", "T12", "T13"],
    metric_sources=comparison_metric_sources,
    filters={"policy": "online_ant_colony"},
)
display(aco_comparison)

,scenario_id,batch,elapsed_s_T11,elapsed_s_T12,elapsed_s_T13,carb_saving_T11,carb_saving_T12,carb_saving_T13,error_T11,error_T12,error_T13,solver_ms_T11,solver_ms_T12,solver_ms_T13,carb_cost_T11,carb_cost_T12,carb_cost_T13
0,scen_1500_92_sine_IP_L,1,10.789,10.883,10.731,50.014356,45.990891,49.600202,3.855406,3.948111,3.856697,0.203392,0.198371,0.129207,126767.856345,140970.605840,128263.518752
1,scen_1500_92_sine_IP_L,8,13.478,7.738,4.671,51.733883,52.461074,50.729540,3.814236,3.811527,3.894034,17.697190,8.737613,1.988514,120577.549121,116775.046236,126354.587215
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25,scen_2000_120_sine_IP_L,2000,127.093,51.411,13.642,55.845632,53.177561,58.436050,3.926252,3.943700,3.833259,1440.850199,570.312501,141.814992,170714.264281,181304.436422,149912.123587
26,scen_2000_120_sine_IP_L,3000,152.052,61.870,15.650,54.289457,52.891158,55.167410,3.848167,3.964716,3.833064,1836.835313,748.585617,179.976197,173723.573595,182468.456022,161552.963258


A higher number of ants and of iterations does not seem to imply more saving. The overhead instead, is significative.\
Hence, we stick to a low number of ants/iterations.

We may stick to batch size = 8

## Comparing ACO, MBA, Hazard and singleton_best


### scen_1500_92

In [69]:
df_on = df[(df["run_id"] == "T10")]
pd.set_option("display.max_rows", 28)
# display(df_on.sort_values(by=SORTING_CRITERIA))
df_92 = df_on[df_on["scenario_id"] == "scen_1500_92"]
display(df_92.sort_values(by=SORTING_CRITERIA))

,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
40,T10,scen_1500_92,105723,104045,8,online_ant_colony,49.262817,4.622,3.891874,130133.339424,0,20,1.994317,20,17.118457,256485.147770,T10_sine_IP_L_online_0_1113_2330_20/scen_1500_...
64,T10,scen_1500_92,105723,98454,1,online_bandit,50.738682,10.195,3.838620,120190.695148,0,20,0.006489,20,18.036359,243985.948614,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_1...
67,T10,scen_1500_92,105723,101006,8,online_bandit,45.964754,3.284,3.813214,136610.555536,0,20,0.046308,20,11.425054,252817.496619,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_1...
69,T10,scen_1500_92,105723,102421,100,online_bandit,48.372054,1.922,3.814525,132146.754816,0,20,0.814600,20,8.730282,255959.735979,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_1...
70,T10,scen_1500_92,105723,104956,250,online_bandit,45.897563,1.785,3.804523,140930.202615,0,20,1.791592,13,5.180909,260487.716952,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_1...
71,T10,scen_1500_92,105723,105197,500,online_bandit,51.142744,1.723,3.849159,127421.066345,0,20,3.252258,7,2.974822,260802.746692,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_1...
72,T10,scen_1500_92,105723,99756,1000,online_bandit,47.268702,1.689,3.848340,130222.117208,0,20,5.253364,4,1.600378,246954.129920,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_1...
73,T10,scen_1500_92,105723,98106,1500,online_bandit,47.521237,1.708,3.816382,127877.624451,0,20,5.835930,3,1.242068,243674.995680,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_1...
74,T10,scen_1500_92,105723,98051,2000,online_bandit,48.809732,1.822,3.797004,125097.645641,0,20,7.125229,2,1.088862,244377.792073,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_1...
75,T10,scen_1500_92,105723,97707,3000,online_bandit,48.713147,1.830,3.794099,124234.925164,0,20,7.558089,2,1.002694,242235.422338,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_1...


### scen_1800_230

In [70]:
# pd.set_option("display.max_rows", 28)
# display(df_on.sort_values(by=SORTING_CRITERIA))
df_230 = df_on[df_on["scenario_id"] == "scen_1800_230"]
display(df_230.sort_values(by=SORTING_CRITERIA))

,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
141,T10,scen_1800_230,128171,126335,8,online_ant_colony,53.253764,5.232,3.908715,147127.201299,0,20,1.763743,20,17.346429,314735.929042,T10_sine_IP_L_online_0_1113_2330_20/scen_1800_...
164,T10,scen_1800_230,128171,122211,1,online_bandit,55.237031,13.341,3.877965,135672.284588,0,20,0.006534,20,18.289354,303090.451744,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_1...
168,T10,scen_1800_230,128171,123727,8,online_bandit,40.514440,3.642,3.874963,186486.733627,0,20,0.043087,20,12.818874,313499.166881,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_1...
169,T10,scen_1800_230,128171,122196,100,online_bandit,48.048111,2.055,3.796503,160697.003328,0,20,0.692522,19,8.328768,309318.885405,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_1...
170,T10,scen_1800_230,128171,123177,250,online_bandit,40.759673,1.841,3.795047,184879.433411,0,20,1.877178,14,5.860526,312083.751138,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_1...
171,T10,scen_1800_230,128171,126231,500,online_bandit,42.721629,1.730,3.788253,182082.342828,0,20,2.685542,8,3.455899,317890.223285,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_1...
172,T10,scen_1800_230,128171,120032,1000,online_bandit,51.510847,1.755,3.847245,146005.702476,0,20,4.628149,4,1.925071,301110.025717,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_1...
173,T10,scen_1800_230,128171,119958,1500,online_bandit,44.659434,1.673,3.789472,166530.664609,0,20,5.382949,3,1.401294,300919.698854,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_1...
174,T10,scen_1800_230,128171,122633,2000,online_bandit,45.728403,1.752,3.799528,168462.820959,0,20,6.126186,3,1.165153,310406.971297,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_1...
175,T10,scen_1800_230,128171,122621,3000,online_bandit,43.113285,1.845,3.808503,176502.513875,0,20,8.334732,2,1.019997,310270.182577,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_1...


### scen_2000_120

In [71]:
df_on = df[(df["run_id"] == "T10")]
pd.set_option("display.max_rows", 28)
# display(df_on.sort_values(by=SORTING_CRITERIA))
df_120 = df_on[df_on["scenario_id"] == "scen_2000_120"]
display(df_120.sort_values(by=SORTING_CRITERIA))

,run_id,scenario_id,reqs_in,reqs_out,batch,policy,carb_saving,elapsed_s,error,carb_cost,reqs_fallback_assigned,paral,solver_ms,peak_concurrent_workers,avg_concurrent_workers,baseline_carb_cost,path
241,T10,scen_2000_120,146276,142462,8,online_ant_colony,53.583901,5.552,3.904392,173559.028766,0,20,1.601804,20,17.099625,373919.898435,T10_sine_IP_L_online_0_1113_2330_20/scen_2000_...
264,T10,scen_2000_120,146276,145265,1,online_bandit,51.925500,16.549,3.869463,185596.104366,0,20,0.006463,20,18.446621,386059.350964,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_2...
267,T10,scen_2000_120,146276,138929,8,online_bandit,51.315932,4.380,3.877556,177929.731388,0,20,0.042221,20,12.948549,365478.355101,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_2...
269,T10,scen_2000_120,146276,138998,100,online_bandit,48.259794,2.021,3.781421,192854.694965,0,20,0.499584,20,8.322044,372736.616693,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_2...
270,T10,scen_2000_120,146276,143140,250,online_bandit,37.031215,1.793,3.816840,240357.985205,0,20,1.499518,17,6.266653,381709.741337,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_2...
271,T10,scen_2000_120,146276,142396,500,online_bandit,36.337830,1.758,3.812168,242603.580934,0,20,2.679694,9,3.949498,381079.661455,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_2...
272,T10,scen_2000_120,146276,139749,1000,online_bandit,43.361832,1.735,3.820278,212104.098698,0,20,4.082268,5,2.110459,374489.686521,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_2...
273,T10,scen_2000_120,146276,136330,1500,online_bandit,49.292807,1.773,3.801419,184724.473938,0,20,5.310431,4,1.528262,364296.391029,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_2...
274,T10,scen_2000_120,146276,137666,2000,online_bandit,43.531604,1.962,3.808293,208065.016559,0,20,5.928615,3,1.242900,368462.773573,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_2...
275,T10,scen_2000_120,146276,135479,3000,online_bandit,48.628951,1.903,3.776655,186615.468010,0,20,6.549335,5,1.095258,363269.722162,T10_sine_IP_L_perrequest_0_1113_2330_20/scen_2...


### singleton_best is our winner

Singleton_best outperforms others.

High batch_size may yield slightly faster results, but make sense only with very high load. They do not seem to imply better savings (actually worse, typically).

We may consider batch_size = 8 the best fit, for both speed and achieved saving.


Singleton_best gets us the best carbon savings.
Hazard almost gets there, but singleton_best  can make fine grained adjustments, while hazard simply blindly follows probabilities.
Bandit and ACO lag begind by 25%, which is considerable.

## T14 - Varying parallelism for our tuned policies

In [72]:
comparison_parallelisms = [1, 10, 20]
t14_parallelism_comparison = compare_metrics_wide(
    df_all,
    row_keys=["scenario_id", "batch", "policy"],
    comparison_column="paral",
    comparison_values=comparison_parallelisms,
    metric_sources=comparison_metric_sources,
    filters={"run_id": "T14"},
    labels={1: "P1", 10: "P10", 20: "P20"},
    key_renames={"batch": "batch_size"},
)
display(t14_parallelism_comparison)

,scenario_id,batch_size,policy,elapsed_s_P1,elapsed_s_P10,elapsed_s_P20,carb_saving_P1,carb_saving_P10,carb_saving_P20,error_P1,error_P10,error_P20,solver_ms_P1,solver_ms_P10,solver_ms_P20,carb_cost_P1,carb_cost_P10,carb_cost_P20
0,scen_1500_92_sine_IP_L,8,online_ant_colony,13.656,5.000,4.835,52.516907,50.921240,50.775329,3.989975,3.879190,3.841224,0.838445,1.801890,2.078546,123864.794353,123869.934946,125465.428783
1,scen_1500_92_sine_IP_L,8,online_bandit,3.029,3.119,3.160,45.948229,43.446332,43.658000,3.791343,3.800836,3.814286,0.035973,0.040164,0.040922,132646.170840,144224.780121,143246.759496
2,scen_1500_92_sine_IP_L,8,online_hazard,2.919,3.097,3.398,74.834967,74.836624,74.828425,3.999454,3.999195,3.999364,0.010020,0.010722,0.011788,63033.745666,63020.481164,63036.431669
3,scen_1500_92_sine_IP_L,8,online_singleton_best,2.763,2.990,3.328,78.110563,77.910686,77.661753,3.997761,3.991847,3.986636,0.002654,0.003769,0.003896,52177.912780,52691.989929,53306.598510
4,scen_1800_230_sine_IP_L,8,online_ant_colony,15.394,5.660,4.786,55.613596,54.913248,51.473263,3.968086,3.853355,3.863037,0.777045,1.665819,1.580342,141260.393367,136698.920400,151981.922185
5,scen_1800_230_sine_IP_L,8,online_bandit,3.601,3.654,3.498,55.290682,50.907993,46.034676,3.873685,3.820424,3.821213,0.037350,0.041560,0.039188,135521.928923,146117.866285,170482.492107
6,scen_1800_230_sine_IP_L,8,online_hazard,3.220,3.576,3.532,74.125593,74.139354,74.109858,3.999804,3.999563,3.999400,0.009538,0.010406,0.010281,79761.335812,79699.936368,79781.496095
7,scen_1800_230_sine_IP_L,8,online_singleton_best,3.228,3.426,3.475,77.281806,76.885055,76.877865,3.995254,3.987548,3.984090,0.002795,0.003328,0.003616,66743.368637,67961.393744,68009.548032
8,scen_2000_120_sine_IP_L,8,online_ant_colony,18.932,6.017,5.297,50.726023,53.351880,52.468527,3.957276,3.887784,3.824658,0.859231,1.478557,1.479880,190638.516308,173740.631469,174980.834506
9,scen_2000_120_sine_IP_L,8,online_bandit,4.120,4.172,3.868,50.489591,49.042546,50.024369,3.780361,3.810636,3.912212,0.035929,0.042619,0.038120,180006.911491,191979.010948,193540.724020


### Parallelism is good for ACO
Parallelism helps speed up ACO runs, not other policies. It neglectably degrades performance.